# Mr. Chicken - Lip-Sync com Wav2Lip (Google Colab)

Este notebook foi gerado pelo **Mr. Chicken** para permitir a sincronização labial gratuita usando o processamento em GPU do Google Colab.

### **Instruções de Uso:**
1. Vá no menu superior e clique em **Ambiente de execução** -> **Alterar tipo de ambiente de execução** e certifique-se de que a **GPU** (T4 GPU) está selecionada.
2. Execute as células abaixo em ordem clicando no botão **Play [ ]** à esquerda de cada célula.
3. Faça o upload do **Avatar** e do **Áudio de Voz** quando solicitado.
4. Aguarde o processamento terminar e faça o download do arquivo `result.mp4` gerado.
5. Envie o arquivo final de volta no painel do Mr. Chicken para concluir a renderização!

### **Passo 1: Instalar e configurar o Wav2Lip**

In [ ]:
# 1. Clonar o repositório Wav2Lip oficial
!git clone https://github.com/Rudrabha/Wav2Lip.git
%cd Wav2Lip

# 2. Baixar os pesos pré-treinados do Wav2Lip+GAN de um espelho estável do Hugging Face
!mkdir -p checkpoints
!wget -q "https://huggingface.co/spaces/sczhou/CodeFormer/resolve/main/weights/facelib/wav2lip_gan.pth" -O checkpoints/wav2lip_gan.pth

# 3. Corrigir incompatibilidades de bibliotecas antigas com o NumPy moderno (erro de np.int)
import fileinput
import sys
for line in fileinput.input("audio.py", inplace=True):
    sys.stdout.write(line.replace("np.int", "int"))

print("✅ Repositório clonado e corrigido com sucesso!")

### **Passo 2: Baixar requerimentos adicionais**

In [ ]:
# Instalar requisitos do Wav2Lip
!pip install -q -r requirements.txt
# Instalar librosa antigo que é compatível com Wav2Lip ou ajustar
!pip install -q librosa==0.8.0
print("✅ Dependências instaladas com sucesso!")

### **Passo 3: Fazer Upload do Avatar e do Áudio do Mr. Chicken**
Execute a célula abaixo. Ela abrirá caixas de seleção de arquivos. Selecione:
1. O arquivo do avatar (imagem `.jpg`/`.png` ou vídeo `.mp4`).
2. O arquivo de áudio de voz gerado (`.wav` ou `.mp3`).

In [ ]:
from google.colab import files
import os
import shutil

# Limpar uploads anteriores se houver
for f in ['input_avatar', 'input_audio']:
    for ext in ['.png', '.jpg', '.jpeg', '.webp', '.mp4', '.avi', '.mov', '.wav', '.mp3']:
        if os.path.exists(f + ext):
            os.remove(f + ext)

print("1. FAÇA UPLOAD DO AVATAR (Vídeo ou Imagem):")
up_avatar = files.upload()
avatar_name = list(up_avatar.keys())[0]
avatar_ext = os.path.splitext(avatar_name)[1].lower()
shutil.move(avatar_name, "input_avatar" + avatar_ext)

print("\n2. FAÇA UPLOAD DO ÁUDIO DE VOZ:")
up_audio = files.upload()
audio_name = list(up_audio.keys())[0]
audio_ext = os.path.splitext(audio_name)[1].lower()
shutil.move(audio_name, "input_audio" + audio_ext)

# Converter áudio para .wav para garantir 100% de compatibilidade
!ffmpeg -y -i "input_audio{audio_ext}" -ac 1 -ar 16000 input_audio_processed.wav

print("\n✅ Arquivos carregados e preparados!")

### **Passo 4: Executar a Sincronização Labial (Lip-Sync)**
Execute esta célula para rodar o modelo Wav2Lip.

In [ ]:
import glob

# Identificar arquivo de avatar
avatar_files = glob.glob("input_avatar.*")[0]
avatar_ext = os.path.splitext(avatar_files)[1].lower()
is_image = avatar_ext in ['.png', '.jpg', '.jpeg', '.webp']

cmd = f"python inference.py --checkpoint_path checkpoints/wav2lip_gan.pth --face {avatar_files} --audio input_audio_processed.wav --outfile result.mp4"
if is_image:
    cmd += " --static"

print(f"Executando comando: {cmd}")
!{cmd}

if os.path.exists("result.mp4"):
    print("\n✅ Lip-sync concluído com sucesso! result.mp4 gerado.")
else:
    print("\n❌ Falha ao gerar o vídeo. Verifique se o rosto no avatar está bem visível e frontal.")

### **Passo 5: Fazer o Download do Vídeo Final**

In [ ]:
if os.path.exists("result.mp4"):
    print("Iniciando download do vídeo...")
    files.download("result.mp4")
else:
    print("Arquivo result.mp4 não encontrado.")